# Lección 5.3 — K vecinos más cercanos (KNN): "dime con quién andas"

> **Módulo 5 · Modelos clásicos supervisados** · Nivel 2 — Aprendiz
> ⏱️ Tiempo estimado: 100–120 min (lección con ejercicios intercalados ≈ 75 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 2.3 (escalado), Lección 2.8 (maldición de la dimensionalidad), Lección 4.4 (sobreajuste vs. subajuste), Lección 4.6 (validación cruzada)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar con una analogía cómo decide KNN y por qué se dice que "no aprende".
- Calcular a mano una distancia y una predicción por votación.
- Explicar por qué KNN **necesita escalar** los datos y cómo hacerlo sin fuga con un **pipeline**.
- Predecir el efecto de sus hiperparámetros (`n_neighbors`, `weights`, `metric`).
- Reconocer cuándo KNN es una buena elección y cuándo no.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.datasets import make_moons, load_wine, load_breast_cancer

plt.rcParams["figure.figsize"] = (8, 4.5)
print("✅ Todo listo")

---
## 1. 📖 La intuición: "dime con quién andas y te diré quién eres"

Te mudas a una ciudad nueva y quieres saber cuánto vale tu departamento. ¿Qué haces? Preguntas a **tus vecinos más parecidos**: los del mismo edificio, del mismo tamaño. Si los 5 departamentos más parecidos se vendieron en torno a 2 millones, el tuyo probablemente también.

**KNN (*k-nearest neighbors*, k vecinos más cercanos) hace exactamente eso:**
1. Llega un ejemplo nuevo.
2. Busca los **k** ejemplos de entrenamiento **más parecidos** (los más cercanos).
3. **Clasificación:** votan, y gana la clase mayoritaria. **Regresión:** se promedian sus valores.

No hay fórmula, ni árbol, ni coeficientes: **el modelo son los propios datos**.

### ✋ Ejercicio 1 — La votación
Llega una fruta misteriosa ❓. Estos son sus 7 vecinos más cercanos, **ordenados del más cercano al más lejano**:

```
  1º 🍋   2º 🍋   3º 🍎   4º 🍎   5º 🍎   6º 🍋   7º 🍎
```

¿Qué predice KNN con **k = 1**, **k = 3**, **k = 5** y **k = 7**?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

- **k = 1** → 🍋 (solo cuenta el más cercano).
- **k = 3** → 🍋 (2 🍋 contra 1 🍎).
- **k = 5** → 🍎 (3 🍎 contra 2 🍋).
- **k = 7** → 🍎 (4 🍎 contra 3 🍋).

**¡La respuesta cambia según k!** Por eso `k` es el hiperparámetro más importante de KNN. Fíjate también en que usamos valores **impares**: con dos clases, así nunca hay empate.
</details>

---
## 2. 📖 El concepto: medir qué tan "cerca" está algo

"Cercano" significa **parecido**, y se mide con una **distancia**. La más común es la **distancia euclidiana**: la de una regla sobre un mapa.

$$\text{distancia} = \sqrt{(\text{diferencia en } x_1)^2 + (\text{diferencia en } x_2)^2 + \dots}$$

Con dos características es el teorema de Pitágoras: la diagonal de un rectángulo.

Probemos con frutas descritas por su **largo** y su **ancho** (en cm):

In [ ]:
frutas = pd.DataFrame({
    "largo": [7.0, 7.5, 8.0, 9.5, 10.0, 11.0],
    "ancho": [7.0, 7.5, 6.5, 6.0, 5.5,  6.0],
    "fruta": ["🍎", "🍎", "🍎", "🍋", "🍋", "🍋"],
}, index=list("ABCDEF"))
frutas

### ✋ Ejercicio 2 — Calcula a mano
Llega una fruta nueva de **largo 9** y **ancho 6.5**.
1. Calcula su distancia a la fruta **C** (8, 6.5) y a la **D** (9.5, 6).
2. ¿Qué predecirá KNN con **k = 3**?

Pista: $\sqrt{a^2 + b^2}$. Luego comprueba con la celda.

✍️ *Mis cálculos:*

In [ ]:
nueva = np.array([9.0, 6.5])
frutas["distancia"] = np.sqrt(((frutas[["largo", "ancho"]] - nueva) ** 2).sum(axis=1)).round(2)
print(frutas.sort_values("distancia"))

K = 3     # ✏️ prueba k = 1 y k = 5
knn_frutas = KNeighborsClassifier(n_neighbors=K).fit(frutas[["largo", "ancho"]], frutas.fruta)
print(f"\nPredicción con k={K}:", knn_frutas.predict(pd.DataFrame({"largo": [9.0], "ancho": [6.5]}))[0])

<details><summary><b>👀 Ver respuesta</b></summary>

1. A **C**: $\sqrt{1^2 + 0^2} = 1.0$. A **D**: $\sqrt{0.5^2 + 0.5^2} ≈ 0.71$.
2. Los 3 más cercanos son **D** (0.71, 🍋), **C** (1.0, 🍎) y **E** (1.41, 🍋) → 2 🍋 contra 1 🍎 → **🍋**.
</details>

---
## 3. 📖 Cómo "aprende": no aprende (es perezoso)

Los demás modelos del módulo **resumen** los datos en algo pequeño: una recta, un árbol, unos coeficientes. KNN no resume nada. Su "entrenamiento" (`fit`) consiste en **guardar** todos los ejemplos. Todo el trabajo ocurre al **predecir**: calcular la distancia del ejemplo nuevo a todos los guardados.

Por eso se le llama **aprendizaje perezoso** (*lazy learning*) o **basado en instancias**.

| | Modelos "ansiosos" (lineal, árbol…) | KNN (perezoso) |
|---|---|---|
| Entrenar | Lento (busca parámetros) | Instantáneo (solo guarda) |
| Predecir | Rápido (aplica la fórmula) | Lento (compara con todos) |
| Memoria | Pequeña | Guarda **todo** el conjunto de datos |

> 💡 **Diferencia con la Lección 3.3:** KNN no tiene función de pérdida ni descenso de gradiente. No tiene parámetros que ajustar, solo hiperparámetros.

### 🔮 Ejercicio 3 — Examen con el libro abierto
Entrenamos un KNN con **k = 1** y le pedimos que prediga… **los mismos datos de entrenamiento**. ¿Qué exactitud esperas? ¿Y en prueba?

✍️ *Mi predicción:*

In [ ]:
X_lunas, y_lunas = make_moons(n_samples=400, noise=0.35, random_state=1)
X_tr, X_te, y_tr, y_te = train_test_split(X_lunas, y_lunas, test_size=0.3, random_state=1)

knn1 = KNeighborsClassifier(n_neighbors=1).fit(X_tr, y_tr)
print(f"k=1 → entrenamiento: {knn1.score(X_tr, y_tr):.0%}  ·  prueba: {knn1.score(X_te, y_te):.0%}")

<details><summary><b>👀 Ver respuesta</b></summary>

**100 % en entrenamiento**, garantizado: el vecino más cercano de cada punto de entrenamiento… ¡es él mismo (distancia 0)! En prueba baja bastante (≈82 %). Es un examen con el libro abierto: la exactitud en entrenamiento de un KNN con k=1 **no dice nada**. Es el sobreajuste más extremo posible, igual que el árbol sin límites de la Lección 5.4.
</details>

---
## 4. 📖 ¿Qué pasa si no escalas los datos?

Este es **el** error clásico con KNN. Las distancias mezclan todas las características, y la que tenga **números más grandes** domina.

### 🔮 Ejercicio 4 — ¿Quién se parece más a Ana?
| Persona | Edad | Ingreso mensual |
|---|---|---|
| Ana | 25 | 30,000 |
| Beto | **60** | 30,500 |
| Carla | **26** | 33,000 |

Para un humano, Carla se parece mucho más a Ana (casi la misma edad, ingreso parecido). ¿Qué dirá la distancia euclidiana?

✍️ *Mi predicción:*

In [ ]:
personas = pd.DataFrame({"edad": [25, 60, 26], "ingreso": [30_000, 30_500, 33_000]},
                        index=["Ana", "Beto", "Carla"])
ana = personas.loc["Ana"]
print("Distancia a Ana SIN escalar:")
print(np.sqrt(((personas - ana) ** 2).sum(axis=1)).round(0))

<details><summary><b>👀 Ver respuesta</b></summary>

Para la distancia, **Beto** (≈ 502) está mucho más cerca que **Carla** (≈ 3,000). La diferencia de 35 años de edad apenas cuenta frente a 500 pesos de ingreso, ¡porque los pesos son números miles de veces más grandes que los años!

La solución es **escalar** (Lección 2.3): poner todas las características en una escala comparable. Con `StandardScaler`, cada columna queda con promedio 0 y desviación estándar 1, así que "1 unidad" significa lo mismo para todas.
</details>

### 📖 Escalar sin hacer trampa: el pipeline
Hay una trampa sutil: si escalas **todos** los datos antes de dividirlos, el escalador "ve" el promedio y la desviación del conjunto de prueba. Es una pequeña **fuga de datos** (Lección 2.6).

La solución es un **pipeline**: una cadena de pasos que se comporta como un solo modelo.

```python
modelo = make_pipeline(StandardScaler(), KNeighborsClassifier())
```

- Al llamar `modelo.fit(X_train, y_train)`, el escalador aprende el promedio y la desviación **solo del entrenamiento**, y luego entrena el KNN.
- Al llamar `modelo.predict(X_nuevo)`, aplica **la misma** transformación y predice.
- En `cross_val_score`, el escalado se **reaprende en cada pliegue**: cero fuga.

> 💡 Los pipelines los usarás en todos los modelos que necesitan escalar (logística, KNN, SVM). En la Lección 7.5 los verás a fondo.

Comprobémoslo con datos reales: 178 vinos de 3 productores, con 13 mediciones químicas. Algunas columnas van de 0.1 a 0.7; la `proline`, de 278 a 1,680.

In [ ]:
Xw, yw = load_wine(return_X_y=True, as_frame=True)
print(Xw.describe().loc[["min", "max"]].T.round(2).head(13))

### 🔮 Ejercicio 5 — ¿Cuánto importa escalar?
Comparamos el mismo KNN (k = 5) con y sin escalar, usando validación cruzada. ¿Cuántos puntos de exactitud crees que hay de diferencia: 1, 5 o más de 20?

✍️ *Mi apuesta:*

In [ ]:
sin_escalar = KNeighborsClassifier(n_neighbors=5)
con_escalar = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))

print(f"Sin escalar: {cross_val_score(sin_escalar, Xw, yw, cv=5).mean():.1%}")
print(f"Con escalar: {cross_val_score(con_escalar, Xw, yw, cv=5).mean():.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

¡Más de **25 puntos** (≈69 % → ≈95 %)! Sin escalar, el KNN prácticamente solo mira la `proline` (y un poco el magnesio): las otras 11 mediciones no cuentan. Escalando, **todas** participan en la distancia.

Mismo modelo, mismo `k`, mismos datos. Solo una línea de diferencia.
</details>

---
## 5. 📖 Hiperparámetros: ¿a cuántos vecinos escucho y cómo?

| Hiperparámetro | Qué controla | Si lo **subes** ⬆️ | Si lo **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| `n_neighbors` (k) | Cuántos vecinos votan | Frontera más suave → riesgo de **subajuste** | Frontera más irregular → riesgo de **sobreajuste** | 5; probar 1–50 con validación cruzada (impar si hay 2 clases) |
| `weights` | Cuánto pesa cada voto | — | — | `uniform` (todos igual) o `distance` (los más cercanos pesan más) |
| `metric` / `p` | Cómo se mide la distancia | — | — | Euclidiana (`p=2`); Manhattan (`p=1`) con muchas características o atípicos |

> 🧠 **Ojo, es al revés que un árbol:** en `max_depth`, más es más complejo. En KNN, **un k más grande es un modelo más simple**: escuchar a muchos vecinos promedia el ruido.

### 🔮 Ejercicio 6 — Pocos o muchos vecinos
Vamos a dibujar la frontera de decisión con **k = 1, 15 y 150** en los datos de las lunas (280 puntos de entrenamiento).
- ¿Cuál tendrá islas diminutas alrededor de puntos sueltos?
- ¿Cuál será demasiado "suave" y se perderá la forma de las lunas?

✍️ *Mis predicciones:*

In [ ]:
def dibujar_frontera(modelo, X, y, ax, titulo):
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - .5, X[:, 0].max() + .5, 250),
                         np.linspace(X[:, 1].min() - .5, X[:, 1].max() + .5, 250))
    zz = modelo.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=.3, cmap="coolwarm")
    ax.scatter(*X.T, c=y, cmap="coolwarm", edgecolor="k", s=15)
    ax.set_title(titulo, fontsize=10); ax.set_xticks([]); ax.set_yticks([])

def comparar(parametro, valores, **fijos):
    fig, axes = plt.subplots(1, len(valores), figsize=(4 * len(valores), 3.8))
    for ax, v in zip(axes, valores):
        m = KNeighborsClassifier(**{parametro: v}, **fijos).fit(X_tr, y_tr)
        dibujar_frontera(m, X_tr, y_tr, ax,
                         f"{parametro}={v}\ntrain {m.score(X_tr, y_tr):.0%} · test {m.score(X_te, y_te):.0%}")
    plt.tight_layout(); plt.show()

comparar("n_neighbors", [1, 15, 150])

<details><summary><b>👀 Ver respuesta</b></summary>

- **k = 1** → islas alrededor de cada punto ruidoso: **sobreajuste** (100 % en entrenamiento).
- **k = 15** → sigue la forma de las lunas sin memorizar el ruido: el mejor en prueba.
- **k = 150** → con más de la mitad de los datos votando, la frontera se vuelve muy suave y pierde las puntas de las lunas: empieza el **subajuste** (baja en entrenamiento, 90 % → 82 %, y también un poco en prueba).

Si llevaras k al extremo (k = todos los puntos), KNN siempre predeciría la **clase mayoritaria**: la línea base de la Lección 4.7.
</details>

### ✏️ Ejercicio 7 — Encuentra el k justo
La celda dibuja la exactitud en entrenamiento y en **validación cruzada** (solo con el entrenamiento, sin tocar la prueba) para muchos valores de k.

In [ ]:
VALORES_K = range(1, 80, 2)    # ✏️ cambia el rango
train_acc, cv_acc = [], []
for k in VALORES_K:
    m = KNeighborsClassifier(n_neighbors=k)
    train_acc.append(m.fit(X_tr, y_tr).score(X_tr, y_tr))
    cv_acc.append(cross_val_score(m, X_tr, y_tr, cv=5).mean())

plt.plot(list(VALORES_K), train_acc, "o-", ms=3, label="Entrenamiento")
plt.plot(list(VALORES_K), cv_acc, "s-", ms=3, label="Validación cruzada")
plt.xlabel("k (n_neighbors)"); plt.ylabel("Exactitud"); plt.legend(); plt.grid(alpha=.3); plt.show()
print(f"Mejor k en validación cruzada: {list(VALORES_K)[int(np.argmax(cv_acc))]}  ({max(cv_acc):.1%})")

Responde:
1. ¿Por qué la curva de entrenamiento empieza en 1.0 y **baja** al subir k?
2. ¿Qué k elegirías? ¿Por qué no simplemente el que da el máximo exacto?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Con k = 1 cada punto se "vota a sí mismo" (Ejercicio 3). Al subir k, el modelo se simplifica y deja de memorizar: el entrenamiento baja. La gráfica está "al revés" que la de `max_depth` en la Lección 5.4: aquí la **izquierda** es el sobreajuste.
2. Cualquier k de la meseta alta de la curva de validación (más o menos entre 3 y 21, todos ≈ 88–90 %); a partir de ~25 la curva baja poco a poco (subajuste). El máximo exacto (k = 13) depende de unos pocos puntos y es en parte ruido; dentro de la meseta, un k algo mayor es **más estable** (más simple).
</details>

### 📖 `weights="distance"`: no todos los votos valen igual
Con `weights="uniform"` (por defecto), el vecino número 15 vota igual que el número 1. Con `weights="distance"`, cada voto pesa **1 / distancia**: los vecinos muy cercanos mandan.

### 🔮 Ejercicio 8 — Votos con peso
Con `weights="distance"` y k = 15, ¿qué exactitud esperas **en entrenamiento**? Pista: ¿cuánto pesa un vecino que está a distancia 0?

✍️ *Mi predicción:*

In [ ]:
comparar("weights", ["uniform", "distance"], n_neighbors=15)

<details><summary><b>👀 Ver respuesta</b></summary>

**100 % en entrenamiento**: cada punto de entrenamiento es su propio vecino a distancia 0, y su voto pesa "infinito". Igual que con k = 1, la exactitud de entrenamiento deja de ser informativa. En prueba, la diferencia con `uniform` suele ser pequeña: `distance` ayuda sobre todo cuando los datos son densos y poco ruidosos. Pruébalo siempre con validación cruzada.
</details>

### 📖 La métrica de distancia
- **Euclidiana** (`p=2`): la regla sobre el mapa. Es la de por defecto.
- **Manhattan** (`p=1`): como caminar por las calles de una ciudad en cuadrícula: sumas las diferencias de cada característica sin elevar al cuadrado. Le afectan menos las diferencias enormes en una sola característica.

En la práctica, el escalado y la elección de **qué características** usar importan mucho más que la métrica.

---
## 6. 📖 ¿Qué pasa si hay muchas características inútiles?

En la Lección 2.8 viste la **maldición de la dimensionalidad**: con muchas dimensiones, todos los puntos quedan "lejos" de todos, y la idea de "vecino cercano" pierde sentido. KNN es el modelo que **más** la sufre, porque todas las características cuentan igual en la distancia, sean útiles o no.

### 🔮 Ejercicio 9 — Ruido en la distancia
A los datos de las lunas (2 características útiles) les añadimos 0, 5, 20 y 100 columnas de **puro ruido**. ¿Qué crees que le pasa a la exactitud de un KNN escalado con k = 15?

✍️ *Mi predicción:*

In [ ]:
rng = np.random.default_rng(0)
for n_ruido in [0, 5, 20, 100]:         # ✏️ prueba otros valores
    X_ruido = np.hstack([X_lunas, rng.normal(size=(len(X_lunas), n_ruido))])
    acc = cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(15)), X_ruido, y_lunas, cv=5).mean()
    print(f"{n_ruido:>3} columnas de ruido → exactitud CV {acc:.0%}  " + "█" * int(acc * 40))

<details><summary><b>👀 Ver respuesta</b></summary>

La exactitud **baja sin parar**: ≈86 % sin ruido, ≈84 % con 5 columnas, ≈80 % con 20 y ≈67 % con 100. Con 100 columnas de ruido y solo 2 útiles, la distancia está dominada por el ruido y el modelo se acerca a lanzar una moneda (50 %).

Un árbol (Lección 5.4) sufriría mucho menos: simplemente no elegiría las columnas de ruido para sus preguntas. **Con KNN, elegir bien las características es tan importante como elegir k.**
</details>

---
## 7. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| Exactitud mucho peor de lo esperado | Datos **sin escalar**: una característica domina la distancia | `make_pipeline(StandardScaler(), KNeighborsClassifier())` |
| 100 % en entrenamiento y mucho menos en prueba | k muy pequeño (o `weights="distance"`) | Subir k; elegirlo con validación cruzada |
| Siempre predice la clase mayoritaria | k demasiado grande o clases desbalanceadas | Bajar k; `weights="distance"`; revisar las métricas por clase (Lección 4.2) |
| Empeora al añadir características | Maldición de la dimensionalidad | Quitar características inútiles; reducir dimensiones (PCA, Lección 8.4) |
| Predicciones lentísimas en producción | Compara con **todos** los datos guardados | Menos datos, índices aproximados, u otro modelo |
| Categorías codificadas como 1, 2, 3 ("rojo" = 1, "verde" = 2, "azul" = 3) | Inventa distancias falsas ("azul" está "más lejos" de "rojo" que "verde") | Usar one-hot (Lección 2.3) |
| Predicciones planas fuera del rango (regresión) | Igual que los árboles, **no extrapola** | Usar otro modelo si necesitas predecir fuera de rango |

### 🔮 Ejercicio 10 — KNN para regresión
`KNeighborsRegressor` predice el **promedio** del valor de los k vecinos. Lo entrenamos con los precios de casas de **50 a 300 m²** (como la mansión de la Lección 5.4). ¿Qué predecirá para **1,000 m²**?

✍️ *Mi apuesta:*

In [ ]:
m2 = rng.uniform(50, 300, 200)
precio = m2 * 15_000 + rng.normal(0, 200_000, 200)
knn_casas = KNeighborsRegressor(n_neighbors=5).fit(m2.reshape(-1, 1), precio)

for tam in [100, 250, 300, 1_000]:
    print(f"{tam:>5} m² → predicción: ${knn_casas.predict([[tam]])[0]:>12,.0f}   (precio 'real' ≈ ${tam * 15_000:,.0f})")

<details><summary><b>👀 Ver respuesta</b></summary>

Exactamente lo mismo que para una casa de 300 m² (≈ $4.3 millones). Los 5 vecinos de la mansión son las 5 casas más grandes que vio, y KNN promedia sus precios. **KNN no extrapola**, igual que un árbol. Una regresión lineal (Lección 5.1) sí lo haría.
</details>

---
## 8. 📋 Ficha del modelo

| | K vecinos más cercanos (KNN) |
|---|---|
| **Tipo** | Supervisado: clasificación y regresión |
| **Intuición** | "Dime con quién andas": busca los k ejemplos más parecidos y los pone a votar (o promedia sus valores) |
| **Cómo aprende** | No aprende: guarda todos los datos (aprendizaje perezoso) y calcula distancias al predecir |
| **Hiperparámetros clave** | `n_neighbors` (k), `weights`, `metric` |
| **✅ Fortalezas** | Muy simple de entender · Sin supuestos sobre la forma de los datos (fronteras de cualquier forma) · Entrenar es instantáneo · Explica con ejemplos ("se parece a estos 5 casos") |
| **❌ Debilidades** | **Necesita escalar** · Sufre mucho con características inútiles y muchas dimensiones · Lento y pesado al predecir con muchos datos · No extrapola · Probabilidades "a saltos" (proporción de votos) |
| **Úsalo cuando…** | Tienes pocas características bien escogidas, datos de tamaño moderado y quieres una línea base no lineal o explicar con casos parecidos (p. ej. recomendaciones) |
| **Evítalo cuando…** | Hay muchas características (texto, cientos de columnas), millones de filas o necesitas predicciones muy rápidas |

## 9. 🧠 Resumen en 5 puntos
1. KNN predice mirando a los **k ejemplos más parecidos**: votación en clasificación, promedio en regresión.
2. No entrena: **guarda** los datos y hace todo el trabajo al predecir (aprendizaje perezoso).
3. **Escalar es obligatorio**; hazlo dentro de un **pipeline** para que el escalado se aprenda solo con el entrenamiento.
4. **k pequeño sobreajusta, k grande subajusta** (al revés que `max_depth`); elígelo con validación cruzada.
5. Sufre la **maldición de la dimensionalidad**: las características inútiles ahogan a las útiles.

---
## 10. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas**. Estos ejercicios son más largos (≈30 min).

### 10.1 Repaso rápido (opción múltiple)

**1.** Tu KNN tiene 100 % en entrenamiento y 72 % en prueba. ¿Qué cambio tiene **más** sentido?
- a) Bajar k de 3 a 1 · b) Subir k de 1 a 15 · c) Cambiar a `weights="distance"` · d) Quitar el escalado

**2.** ¿Qué hace realmente `KNeighborsClassifier.fit()`?
- a) Calcula coeficientes · b) Construye un árbol · c) Guarda los datos (y prepara una estructura para buscar vecinos) · d) Descenso de gradiente

**3.** Tienes 2 clases con 300 y 700 ejemplos. Pones k = 1,000 (todos los datos). ¿Qué predice siempre?
- a) La clase de 300 · b) La clase de 700 · c) Una clase al azar · d) Da error

**4.** ¿En cuál de estos casos es **más** importante escalar?
- a) Árbol de decisión con edad e ingreso · b) KNN con edad e ingreso · c) KNN con largo y ancho en cm · d) Ninguno

**5.** ¿Por qué el escalador debe ir **dentro** del pipeline al hacer validación cruzada?
- a) Porque es más rápido
- b) Para que el promedio y la desviación se aprendan solo con los datos de entrenamiento de cada pliegue
- c) Porque KNN no acepta datos escalados de otra forma
- d) No importa dónde vaya

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** La brecha es sobreajuste; en KNN se combate **subiendo** k. a) y c) empeoran (o mantienen) la memorización; d) empeora todo.
2. **c)** KNN es perezoso: guarda los datos.
3. **b)** Si votan todos, gana siempre la mayoría: es la línea base "clase mayoritaria".
4. **b)** Edad (decenas) e ingreso (miles) tienen escalas muy distintas y KNN mezcla ambas en una distancia. En c) las dos están en cm. El árbol (a) no necesita escalar.
5. **b)** Si escalas antes de dividir, el escalador "ve" los datos de validación: es una fuga de datos (Lección 2.6).
</details>

### 10.2 Preguntas para pensar (sin ejecutar)

**A.** Netflix quiere recomendar películas: "a los usuarios **parecidos** a ti les gustó X". ¿En qué se parece esto a KNN? ¿Qué problema tendría si hay 200 millones de usuarios y 20,000 películas (columnas)?

**B. Caso de análisis 💼** Un hospital quiere un modelo para clasificar radiografías y otro para clasificar pacientes según 6 análisis de sangre (300 pacientes). Un colega propone KNN para ambos. ¿En cuál tiene más sentido? ¿Qué ventaja tiene KNN para explicarle una predicción a un médico?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Es la idea de KNN aplicada a usuarios: busca los k usuarios más parecidos y recomienda lo que les gustó (filtrado colaborativo, Lección 8.7). Problemas: comparar contigo a **200 millones** de usuarios en cada recomendación es lentísimo, y con **20,000 columnas** casi vacías la maldición de la dimensionalidad hace que todos parezcan igual de lejanos. Por eso en la práctica se reducen dimensiones (*embeddings*, Lección 11.2) y se usan búsquedas aproximadas de vecinos.

**B.** Tiene más sentido para los **análisis de sangre**: pocas características (6), pocos pacientes (300) y valores numéricos que se pueden escalar. Para radiografías, cada píxel sería una característica (cientos de miles) y la distancia entre píxeles casi no refleja el contenido de la imagen: ahí funcionan mucho mejor las redes convolucionales (Módulo 10). Ventaja para el médico: KNN **explica con casos**: *"este paciente se clasificó como de riesgo porque sus 7 pacientes más parecidos lo eran; aquí están sus expedientes"*.
</details>

### 10.3 🧩 Práctica integradora: "Diagnóstico por parecido"

Un laboratorio quiere apoyar el diagnóstico de tumores de mama (569 pacientes, 30 mediciones de las células) con un KNN que, además, muestre al médico **los casos más parecidos**.

Lo harás en 6 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Prepara los datos.** Dejamos 25 % para la prueba final y **no lo tocamos** hasta el Paso 6.

In [ ]:
Xc, yc = load_breast_cancer(return_X_y=True, as_frame=True)
yc = 1 - yc     # 1 = maligno
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.25, stratify=yc, random_state=0)
print(f"Entrenamiento: {len(Xc_tr)} · Prueba: {len(Xc_te)} · Malignos: {yc.mean():.0%}")
print(Xc.describe().loc[["min", "max"], ["mean radius", "mean area", "mean smoothness"]])

**Paso 2 🔮 — Sin escalar.** Mira los rangos de arriba: `mean area` llega a 2,500 y `mean smoothness` no pasa de 0.17. ¿Qué exactitud esperas de un KNN (k = 5) sin escalar comparado con uno escalado?

✍️ *Mi apuesta:*

In [ ]:
for nombre, modelo in [("Sin escalar", KNeighborsClassifier(5)),
                       ("Con escalar", make_pipeline(StandardScaler(), KNeighborsClassifier(5)))]:
    print(f"{nombre}: exactitud CV {cross_val_score(modelo, Xc_tr, yc_tr, cv=5).mean():.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

Sin escalar ≈ 93 %, con escalar ≈ 97 %. La diferencia es menor que con los vinos porque las columnas grandes (área, perímetro) por suerte **sí** son informativas aquí. Pero 4 puntos en un diagnóstico médico son muchos pacientes. **Escalar nunca sobra con KNN.**
</details>

**Paso 3 ✏️ — Elige k con validación cruzada.** Cambia el rango si quieres explorar más.

In [ ]:
VALORES_K = [1, 3, 5, 7, 9, 11, 15, 21, 31, 51]     # ✏️
for k in VALORES_K:
    acc = cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(k)), Xc_tr, yc_tr, cv=5).mean()
    print(f"k={k:>2} → exactitud CV {acc:.3f}  " + "█" * int((acc - 0.85) * 300))

✍️ *Mi k elegido:*

<details><summary><b>👀 Ver respuesta</b></summary>

La exactitud es máxima y casi plana entre **k = 3 y k = 7** (≈ 0.97), y luego baja poco a poco (≈ 0.95 con k = 15–21 y ≈ 0.93 con k = 51: subajuste). Con k = 1 también baja un poco (sobreajuste). Entre los empatados, **k = 7** es el más estable (más simple) y es una elección razonable.
</details>

**Paso 4 ✏️ — ¿`uniform` o `distance`?** Usa el k que elegiste.

In [ ]:
K = 7    # ✏️ tu k
for w in ["uniform", "distance"]:
    acc = cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(K, weights=w)), Xc_tr, yc_tr, cv=5).mean()
    print(f"weights={w:<8} → exactitud CV {acc:.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Empatan (≈ 0.970 las dos). Cuando dos opciones empatan, elige la más simple: `uniform`. Lo importante es haberlo **comprobado** con validación cruzada y no con la prueba.
</details>

**Paso 5 ▶️ — Explicar con casos parecidos.** Esta es la gran ventaja de KNN. Tomamos un paciente de prueba y mostramos sus vecinos más cercanos del entrenamiento.

In [ ]:
final = make_pipeline(StandardScaler(), KNeighborsClassifier(K)).fit(Xc_tr, yc_tr)

PACIENTE = 4         # ✏️ prueba con otros pacientes (0 a 142)
x = Xc_te.iloc[[PACIENTE]]
distancias, indices = final[-1].kneighbors(final[0].transform(x))
vecinos = Xc_tr.iloc[indices[0]][["mean radius", "mean texture", "mean concave points"]].copy()
vecinos["diagnóstico"] = np.where(yc_tr.iloc[indices[0]] == 1, "maligno", "benigno")
vecinos["distancia"] = distancias[0].round(2)

print(f"Paciente {PACIENTE}: probabilidad de maligno = {final.predict_proba(x)[0, 1]:.0%} "
      f"(diagnóstico real: {'maligno' if yc_te.iloc[PACIENTE] == 1 else 'benigno'})")
print(x[["mean radius", "mean texture", "mean concave points"]].round(3).to_string())
vecinos

✋ ¿Cómo le explicarías esta predicción al médico en una frase? Fíjate también en la "probabilidad": ¿qué valores puede tomar con k = 7?

✍️ *Mi explicación:*

<details><summary><b>👀 Ver respuesta</b></summary>

Con k = 7, el paciente 4 tiene **4 vecinos malignos y 3 benignos** → "probabilidad" de 57 % → el modelo dice *maligno*… y se equivoca: era benigno. Es un caso **dudoso**, y KNN lo deja ver: *"De los 7 pacientes de nuestro historial más parecidos a este, 4 tenían tumor maligno y 3 benigno; aquí están sus mediciones para que usted las compare."* Un médico sabría que este caso pide más estudios, en lugar de confiar a ciegas en el "maligno".

La "probabilidad" de KNN es solo la **proporción de votos**: con k = 7 solo puede valer 0/7, 1/7, 2/7… 7/7. Es una probabilidad "a saltos", menos fina que la de una regresión logística (Lección 5.2).
</details>

**Paso 6 ▶️ — Examen final, una sola vez.**

In [ ]:
print(f"KNN (k={K}) en prueba: {final.score(Xc_te, yc_te):.1%}")
print(f"Modelo 'todos benignos': {1 - yc_te.mean():.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

≈ **93 %**, un poco por debajo del ≈97 % de la validación cruzada: con solo 143 pacientes de prueba, cada error mueve la exactitud 0.7 puntos, así que esa diferencia son unos 5 pacientes (es normal que haya algo de variación). El modelo "todos benignos" se queda en ≈63 %. Un KNN escalado, con un k bien elegido, es un modelo sencillo y competitivo con pocas características.
</details>

---
## 📚 Para profundizar
- 📺 **Video:** StatQuest, "K-nearest neighbors, Clearly Explained" (inglés, con subtítulos).
- 📘 **Documentación:** scikit-learn, guía de usuario, sección *Nearest Neighbors*, y *Pipelines and composite estimators* (pipelines).
- 🎮 **Interactivo:** busca "KNN interactive demo" (p. ej. la de Stanford CS231n): mueve los puntos y cambia k para ver cómo se mueve la frontera.

---
⬅️ **Anterior:** [5.2 Regresión logística: de la recta a la probabilidad](5.2-regresion-logistica.ipynb) · ➡️ **Siguiente:** [5.4 Árboles de decisión: el juego de las 20 preguntas](5.4-arboles-de-decision.ipynb)